# 5장 2강: 변수 스코프와 lambda 함수

변수 스코프(scope): 변수를 접근할 수 있는 범위

## 1. 전역 변수(Global)와 지역 변수(Local)

- 전역변수(Global): 함수 외부에 정의된 변수, 코드 전역에서 접근 가능
- 지역변수(Local): 함수 내부에 정의된 변수, 함수가 실행될 때만 유효한 변수

In [8]:
a = 10 # 전역 변수
def func():
    b = 20
    print(f"{a=}, {b=}")
    
# print(a, b) # NameError : b
print(a)

func()

10
a=10, b=20


In [ ]:
c = 30 # global 

def outer():
    a = 10
    # print("b:", b) # error
    print("c:", c)

    def inner():
        b = 20
        print(f"{a=}, {b=}, {c=}")

    inner() # inner()는 호출이 되어야지만 존재. 

outer() # outer -> inner O, inner -> outer X (outer scope can reach within its function code)

c: 30
a=10, b=20, c=30


In [ ]:
def outer():
    def inner():
        print("Hello")
    inner() # call inner()


outer()

Hello


## 2. 파이썬의 변수 검색 규칙: LEGB 스코프

접근 우선순위: 좁은 범위가 우선순위 높음(좁은 범위 먼저 실행)

***L(Local - 지역 변수) > E(Enclosing - 외부 함수) > G(Global - 전역 변수) > B(Builtin - 내장 변수)***


```text
Global Scope
│
├── x = 100
│
└── outer()
    │
    └── Local Scope of outer
        ├── x = 10
        └── inner()
            │
            └── Local Scope of inner
                └── x = 1
```

```text
inner()에서 x를 사용 
    ↓
L — Local inner의 Local Scope 
    ↓ 
없으면 E — Enclosing outer의 Local Scope 
    ↓ 
없으면 G — Global 모듈의 Global Scope 
    ↓ 
없으면 B — Builtin Python 내장 이름 
    ↓ 
없으면 NameError
```

In [ ]:
a = True

def outer():
    a = 'Enclosing'
    print("outer:", a)

    def inner():
        # a = 'Local'
        global a # a라는 변수는 전역변수를 찾겠다
        a = not a
        print("inner:", a)

    inner()

print(a)
outer()

outer: Enclosing
inner: False


## 3. global 키워드

`global 변수명`: 전역 공간에 있는 '변수'를 수정하겠다- 함수 내에서 전역 변수를 수정할 수 있는 수정 통로를 열어준다

## 4. 일급 함수(First-class Function)

- '변수=함수'
- 함수와 변수를 동일하게 취급
- 함수의 **매개변수로 함수**를 사용할 수 있다 -- callback 함수
- 함수의 **반환값으로 함수**를 사용할 수 있다 -- 팩토리 함수, 클로저(closure)

In [49]:
def callback():
    print("callback()")


def func(a):
    print("func()")
    a() # 함수 a를 호출

func(callback) # 함수의 반환값으로 함수를 사용, 변수명만 사용

func()
callback()


In [ ]:
def outer(num1):
    def inner(num2):
        return num1 - num2
    
    return inner # inner()를 반환값으로서 내보냄. inner는 first-class function

a = outer(10) 
# outer(num1=10) 실행
# → inner 함수 객체 생성
# → inner가 num1을 참조하므로 num1은 cell로 캡처됨
# → return inner
# → outer()의 실행 Frame은 Call Stack에서 제거됨
# → 하지만 inner가 num1을 계속 참조하므로
#   num1의 값은 Heap영역의 PyCellObject를 통해 유지됨
# → a는 inner 함수 객체를 참조함
print(a(20)) # a(20) = inner(20). 
print(a) # 10 - 20 = -10
b = outer(20) # 
print(b(20))


# outer의 지역변수 2개 = num1, inner (first-class function)

-10
<function outer.<locals>.inner at 0x000002B97B832610>
0


In [66]:
def change_numbers(*args, callback):
    nums = list(args) #tuple -> list로 변환 why?: tuple은 index 접근 조회만 가능, 값 변경 불가

    for i in range(len(nums)):
        nums[i] = callback(nums[i])
    
    return nums


def square(num):
    return num ** 2


change_numbers(10, 20, 30, callback=square) 
# *args 때문에 square도 튜플 안에 들어가는 숫자로 오해할 수 있으므로 kywarg 형식으로 함수 매개변수 작성
# 혹은 change_numbers 매개변수 순서 바꾸기
    


[100, 400, 900]

In [ ]:
def plus(num):
    return num + 10

change_numbers(10, 20, 30, callback=plus) #plus(nums[i])

[20, 30, 40]

**closure**
- outer function 실행 끝난 뒤에도 inner가 outer 변수를 참조하고 접근할 수 있는 함수


## 5. lambda 함수

- `lambda parameter, parameter, ...: code`
- 간단하게 일회용으로 사용하는 경우 한줄의 식으로 간단하게 함수 정의
- True/False 혹은 계산 가능
- 일회용으로서 메모리 절약 가능
- parameter 이름도 한글자로 최대한 간결하게
- 보통 변수에 대입하지 않고 일급함수로서 매개변수로 쓰일 때 바로 써버림
- 변수에 담기지 않으면 소멸

In [63]:
square = lambda num: num ** 2
print(square(2))

4


In [64]:
change_numbers(10, 20, 30, callback= lambda num: num ** 2)

[100, 400, 900]

In [67]:
def change_numbers2(callback, *args):
    numbers = list(args)

    for i in range(len(numbers)):
        numbers[i] = callback(numbers[i])

    return numbers

change_numbers2(lambda x: x **2, 20, 30, 40)

[400, 900, 1600]

## 6. 전역 변수와 지역 변수의 이름 충돌과 은닉 법칙

## 7. lambda와 map(), filter()를 활용한 데이터 일괄 가공

- `map(함수, 시퀀스)`: 변환 작업, **sequence에 있는 모든 item에 함수를 적용**시킴, list()로 결과 보기 가능
- `fliter(함수, 시퀀스)`: **함수 조건**에서 데이터를 선별


**iterables vs. sequence**
- `iterables`: for 문 순회 가능, sequence를 포함하는 개념 ex) list, tuple, str, set ,dict, range

- `sequence`: 순서 O, index O. ex) list, tuple, str, range


map(function, iterables), filter(function, iterables)
- iterbles는 paremeter로서 function의 argument

In [ ]:
# map()
fruits= ['apple', 'banana', 'orange', 'mango' ]
list(map(lambda x: f"**{x}**", fruits))


['**apple**', '**banana**', '**orange**', '**mango**']

In [77]:
# filter()
numbers = [*range(1,11)] # * 사용해서 펼쳐주기
list(filter(lambda x: x % 2 == 1 , numbers))


[1, 3, 5, 7, 9]